# CSV Basics — 06: Character Encodings

## What you will learn
Encoding errors are the #1 silent data corruption issue in CSV pipelines.
A file that "reads successfully" may have completely wrong characters.

In this notebook:
1. UTF-8 vs Latin-1 vs Windows-1252 — the most common encodings
2. BOM (Byte Order Mark) — the invisible prefix that breaks headers
3. Detecting encoding of unknown files
4. Spark's encoding option — reading and writing with correct encoding
5. Handling mixed-encoding files
6. Building an encoding-safe pipeline


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 09:09:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


## Step 1 — What is Character Encoding?

Every text file stores characters as bytes. The encoding defines the mapping.
- **ASCII**: 128 characters, 1 byte each — only English letters, numbers, symbols
- **Latin-1** (ISO-8859-1): 256 characters — adds Western European accented chars (é, ü, ñ)
- **Windows-1252** (CP1252): similar to Latin-1 but different at 0x80-0x9F — very common in Windows exports
- **UTF-8**: handles ALL Unicode characters (100K+), variable 1-4 bytes per char — modern standard


In [2]:
import pathlib

# Create files with different encodings
test_content = "id,name,city\n1,François,Zürich\n2,José,München\n3,Ñoño,São Paulo\n"

# Write as UTF-8 (modern standard)
utf8_path = f"{DATA_DIR}/utf8_sample.csv"
pathlib.Path(utf8_path).write_bytes(test_content.encode('utf-8'))

# Write as Latin-1 (Western European)
latin1_path = f"{DATA_DIR}/latin1_sample.csv"
pathlib.Path(latin1_path).write_bytes(test_content.encode('latin-1', errors='replace'))

# Write as Windows-1252
cp1252_path = f"{DATA_DIR}/cp1252_sample.csv"
pathlib.Path(cp1252_path).write_bytes(test_content.encode('cp1252', errors='replace'))

# Show raw bytes to understand the difference
print("UTF-8 bytes for 'François':")
word = "François"
print(f"  UTF-8  : {word.encode('utf-8').hex()}")
print(f"  Latin-1: {word.encode('latin-1').hex()}")
print(f"  ç in UTF-8 = 2 bytes (0xc3 0xa7)")
print(f"  ç in Latin-1 = 1 byte (0xe7)")
print()
print("This is why reading a Latin-1 file as UTF-8 produces garbage (mojibake)!")

UTF-8 bytes for 'François':
  UTF-8  : 4672616ec3a76f6973
  Latin-1: 4672616ee76f6973
  ç in UTF-8 = 2 bytes (0xc3 0xa7)
  ç in Latin-1 = 1 byte (0xe7)

This is why reading a Latin-1 file as UTF-8 produces garbage (mojibake)!


In [3]:
# Demonstrate the encoding problem
print("1. Read UTF-8 file with correct encoding:")
spark.read.csv(utf8_path, header=True,
               encoding="UTF-8").show(truncate=False)

print("2. Read UTF-8 file with WRONG encoding (Latin-1):")
spark.read.csv(utf8_path, header=True,
               encoding="ISO-8859-1").show(truncate=False)

print("3. Read Latin-1 file with WRONG encoding (UTF-8) — may fail or show garbage:")
try:
    spark.read.csv(latin1_path, header=True,
                   encoding="UTF-8").show(truncate=False)
except Exception as e:
    print(f"  Error: {type(e).__name__}: {str(e)[:80]}")

print("4. Read Latin-1 file with CORRECT encoding:")
spark.read.csv(latin1_path, header=True,
               encoding="ISO-8859-1").show(truncate=False)

1. Read UTF-8 file with correct encoding:


+---+--------+---------+
|id |name    |city     |
+---+--------+---------+
|1  |François|Zürich   |
|2  |José    |München  |
|3  |Ñoño    |São Paulo|
+---+--------+---------+

2. Read UTF-8 file with WRONG encoding (Latin-1):
+---+---------+----------+
|id |name     |city      |
+---+---------+----------+
|1  |FranÃ§ois|ZÃ¼rich   |
|2  |JosÃ©    |MÃ¼nchen  |
|3  |ÃoÃ±o   |SÃ£o Paulo|
+---+---------+----------+

3. Read Latin-1 file with WRONG encoding (UTF-8) — may fail or show garbage:
+---+--------+---------+
|id |name    |city     |
+---+--------+---------+
|1  |Fran�ois|Z�rich   |
|2  |Jos�    |M�nchen  |
|3  |�o�o    |S�o Paulo|
+---+--------+---------+

4. Read Latin-1 file with CORRECT encoding:
+---+--------+---------+
|id |name    |city     |
+---+--------+---------+
|1  |François|Zürich   |
|2  |José    |München  |
|3  |Ñoño    |São Paulo|
+---+--------+---------+



## Step 2 — BOM (Byte Order Mark): The Invisible Enemy

Some tools (Excel, Windows Notepad) add a BOM at the start of UTF-8 files.
The BOM is the invisible sequence `EF BB BF` — 3 bytes before the first character.
This breaks the column name of the first column.


In [4]:
# Create a UTF-8 file WITH BOM (as Excel would export)
bom_content = "\ufeff" + "order_id,customer,amount\n1,Alice,100\n2,Bob,200\n"
bom_path = f"{DATA_DIR}/with_bom.csv"
pathlib.Path(bom_path).write_bytes(bom_content.encode('utf-8-sig'))

# Without BOM handling
df_no_bom_handle = spark.read.csv(bom_path, header=True, encoding="UTF-8")
print("Reading BOM file WITHOUT handling:")
print(f"  Columns: {df_no_bom_handle.columns}")
print(f"  First column name: '{df_no_bom_handle.columns[0]}'")
print("  ← Notice the invisible BOM character at start of 'order_id'!")
df_no_bom_handle.show()

# With UTF-8-BOM encoding
df_bom_handled = spark.read.csv(bom_path, header=True, encoding="UTF-8-BOM")
print("\nReading BOM file WITH UTF-8-BOM encoding:")
print(f"  Columns: {df_bom_handled.columns}")
df_bom_handled.show()

Reading BOM file WITHOUT handling:
  Columns: ['order_id', 'customer', 'amount']
  First column name: 'order_id'
  ← Notice the invisible BOM character at start of 'order_id'!
+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|       1|   Alice|   100|
|       2|     Bob|   200|
+--------+--------+------+



IllegalArgumentException: [INVALID_PARAMETER_VALUE.CHARSET] The value of parameter(s) `charset` in `CSVOptions` is invalid: expects one of the iso-8859-1, us-ascii, utf-16, utf-16be, utf-16le, utf-32, utf-8, but got UTF-8-BOM. SQLSTATE: 22023

## Step 3 — Detecting Unknown Encoding

When you receive a file from an external source, how do you know its encoding?


In [ ]:
# Use chardet (or charset_normalizer) for encoding detection
try:
    import chardet
    HAS_CHARDET = True
except ImportError:
    HAS_CHARDET = False
    print("chardet not available — using manual detection")

def detect_encoding(filepath, sample_bytes=10000):
    """Detect file encoding by reading first N bytes."""
    with open(filepath, 'rb') as f:
        raw = f.read(sample_bytes)

    # Check for BOM first
    if raw.startswith(b'\xef\xbb\xbf'):
        return 'UTF-8-BOM'
    elif raw.startswith(b'\xff\xfe'):
        return 'UTF-16-LE'
    elif raw.startswith(b'\xfe\xff'):
        return 'UTF-16-BE'

    # Try chardet if available
    if HAS_CHARDET:
        result = chardet.detect(raw)
        return f"{result['encoding']} (confidence: {result['confidence']:.0%})"

    # Manual test: try UTF-8 first, then Latin-1
    try:
        raw.decode('utf-8')
        return 'UTF-8 (verified)'
    except UnicodeDecodeError:
        return 'Non-UTF-8 (likely Latin-1 or CP1252)'

for path, label in [(utf8_path, "UTF-8"),
                     (latin1_path, "Latin-1"),
                     (bom_path, "UTF-8 with BOM")]:
    detected = detect_encoding(path)
    print(f"  {label:<20} → detected: {detected}")

## Step 4 — Reading Common Encodings in Spark


In [ ]:
# Common encodings and their Spark names
encoding_map = {
    "UTF-8":         "UTF-8",
    "UTF-8-BOM":     "UTF-8-BOM",
    "Latin-1":       "ISO-8859-1",
    "Windows-1252":  "windows-1252",
    "UTF-16":        "UTF-16",
    "ASCII":         "ASCII",
}

print("Encoding options for spark.read.csv:")
for human_name, spark_name in encoding_map.items():
    print(f"  {human_name:<20} → encoding='{spark_name}'")
print()

# Read Latin-1 and normalize to UTF-8 Parquet
print("Best practice: read with correct encoding, write as UTF-8 Parquet")
df_normalized = spark.read.csv(latin1_path, header=True, encoding="ISO-8859-1")
df_normalized.write.mode("overwrite").parquet(f"{DATA_DIR}/normalized_utf8")
df_read_back = spark.read.parquet(f"{DATA_DIR}/normalized_utf8")
print("Normalized output:")
df_read_back.show(truncate=False)
print("All special characters preserved correctly in Parquet (UTF-8 internally)")

## Step 5 — Encoding-Safe Pipeline

Production rule: standardize all incoming CSV files to UTF-8 immediately.


In [ ]:
def safe_csv_ingest(path, target_encoding="UTF-8"):
    """
    Detect encoding, read correctly, normalize to UTF-8 Parquet.
    Returns a Spark DataFrame with correct characters.
    """
    detected = detect_encoding(path)
    print(f"  Detected encoding: {detected}")

    # Map detected to Spark encoding option
    if 'BOM' in detected:
        spark_encoding = 'UTF-8-BOM'
    elif 'UTF-8' in detected:
        spark_encoding = 'UTF-8'
    elif 'Latin' in detected or 'ISO-8859' in detected:
        spark_encoding = 'ISO-8859-1'
    else:
        spark_encoding = 'UTF-8'  # fallback

    print(f"  Using Spark encoding: {spark_encoding}")
    df = spark.read.csv(path, header=True, encoding=spark_encoding)
    return df

print("Safe ingestion pipeline:")
for path, label in [(utf8_path,"UTF-8"), (latin1_path,"Latin-1"), (bom_path,"BOM")]:
    print(f"\nProcessing {label}:")
    df = safe_csv_ingest(path)
    df.show(truncate=False)

## Summary: Encoding Cheat Sheet

| Encoding | When seen | Spark option | Notes |
|---|---|---|---|
| UTF-8 | Modern files, Linux/Mac | `encoding='UTF-8'` | Default, always prefer |
| UTF-8-BOM | Excel exports, Windows | `encoding='UTF-8-BOM'` | BOM stripped automatically |
| Latin-1 | Old European systems | `encoding='ISO-8859-1'` | Western Europe only |
| Windows-1252 | Windows/Excel old | `encoding='windows-1252'` | Similar to Latin-1 |
| UTF-16 | Windows Unicode | `encoding='UTF-16'` | Rare in CSV |

**Rule:** Always normalize incoming CSV to UTF-8 Parquet as the first pipeline step.
